In [1]:
!pip install transformers datasets evaluate sacrebleu accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("dvgodoy/yoda_sentences")
dataset

README.md:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentences.csv:   0%|          | 0.00/98.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/720 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'translation', 'translation_extra'],
        num_rows: 720
    })
})

In [3]:
dataset["train"][0]

{'sentence': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'translation_extra': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [4]:
dataset["train"].column_names

['sentence', 'translation', 'translation_extra']

In [5]:
dataset_split = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_dataset = dataset_split["train"]
val_dataset = dataset_split["test"]

len(train_dataset), len(val_dataset)

(576, 144)

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google-t5/t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
source_column = "sentence"
target_column = "translation_extra"

max_input_length = 64
max_target_length = 64

def preprocess_function(examples):
    inputs = [
        "translate English to Yoda style: " + text
        for text in examples[source_column]
    ]

    targets = examples[target_column]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_train[0]

Map:   0%|          | 0/576 [00:00<?, ? examples/s]

{'sentence': 'A round hole was drilled through the thin board.',
 'translation': 'Through the thin board, a round hole was drilled.',
 'translation_extra': 'Through the thin board, a round hole was drilled.',
 'input_ids': [13959,
  1566,
  12,
  6545,
  26,
  9,
  869,
  10,
  71,
  1751,
  6356,
  47,
  3,
  28940,
  190,
  8,
  5551,
  1476,
  5,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [4582, 8, 5551, 1476, 6, 3, 9, 1751, 6356, 47, 3, 28940, 5, 1]}

In [11]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-yoda-style",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_steps=10,
    save_total_limit=2,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.715596,0.598396
2,0.520290,0.540643
3,0.392401,0.483939
4,0.307392,0.488399
5,0.372789,0.491195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=360, training_loss=0.5590760396586524, metrics={'train_runtime': 795.4644, 'train_samples_per_second': 3.621, 'train_steps_per_second': 0.453, 'total_flos': 17427371655168.0, 'train_loss': 0.5590760396586524, 'epoch': 5.0})

In [12]:
def generate_yoda(sentence):
    input_text = "translate English to Yoda style: " + sentence

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
test_sentences = [
    "I am ready to study.",
    "You must finish your homework.",
    "The model learns from the data.",
    "This project is difficult but interesting.",
    "I need more coffee today.",
    "The student wrote a report about neural networks.",
    "We trained a model on a small dataset."
]

for sentence in test_sentences:
    print("Input:", sentence)
    print("Yoda:", generate_yoda(sentence))
    print()

Input: I am ready to study.
Yoda: Ready to study, i am.

Input: You must finish your homework.
Yoda: You must finish your homework, you must.

Input: The model learns from the data.
Yoda: From the data, the model learns.

Input: This project is difficult but interesting.
Yoda: Hrrmmm. Hard but interesting, this project is.

Input: I need more coffee today.
Yoda: More coffee today, i need.

Input: The student wrote a report about neural networks.
Yoda: A report about neural networks, the student wrote.

Input: We trained a model on a small dataset.
Yoda: On a small dataset, a model we trained.

